# ETL Pipeline Demo — Bibliometrix Python
## Advanced Level 

This notebook demonstrates the ETL pipeline developed for the Bibliometrix-Python project.
The pipeline extracts data from OpenAlex and PubMed APIs, transforms it into the WoS standard schema, and validates the output.

---
## PHASE 1: EXTRACT
Data is retrieved via REST APIs from OpenAlex and PubMed.
The `retrieve()` function handles pagination, rate limits, and retries automatically.

In [1]:
from www.services.api_retriever import retrieve

print("=== EXTRACT: OpenAlex ===")
records_oa = retrieve(query="machine learning", platform="openalex", total=10)
print(f"Records retrieved: {len(records_oa)}")
print(f"Sample raw keys: {list(records_oa[0].keys())[:8]}")
print(f"\nSample title: {records_oa[0].get('title', 'N/A')}")

=== EXTRACT: OpenAlex ===
Records retrieved: 10
Sample raw keys: ['id', 'doi', 'title', 'display_name', 'relevance_score', 'publication_year', 'publication_date', 'ids']

Sample title: Scikit-learn: Machine Learning in Python


In [2]:
print("=== EXTRACT: PubMed ===")
records_pm = retrieve(query="machine learning", platform="pubmed", total=10)
print(f"Records retrieved: {len(records_pm)}")
print(f"Sample raw keys: {list(records_pm[0].keys())[:8]}")
print(f"\nSample title: {records_pm[0].get('title', 'N/A')}")

=== EXTRACT: PubMed ===
Records retrieved: 10
Sample raw keys: ['uid', 'pubdate', 'epubdate', 'source', 'authors', 'lastauthor', 'title', 'sorttitle']

Sample title: Astrobiology in the Time of Artificial Intelligence.


---
## PHASE 2: TRANSFORM
Raw API responses are mapped to the WoS standard schema using mapping dictionaries.
Multi-value fields are cast to `list[str]`, scalar fields to `str`, and `TC` to `int`.

In [3]:
from www.services.standardizer import standardize
import pandas as pd

print("=== TRANSFORM: OpenAlex ===")
df_oa = standardize(records_oa, source="openalex")
print(f"Shape: {df_oa.shape}")
print(f"Columns: {df_oa.columns.tolist()}")
df_oa[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: OpenAlex ===
Shape: (10, 26)
Columns: ['UT', 'DI', 'TI', 'PY', 'LA', 'DT', 'TC', 'SO', 'JI', 'AU', 'AF', 'C1', 'RP', 'AB', 'VL', 'IS', 'BP', 'EP', 'DE', 'AU_CO', 'CR', 'ID', 'PMID', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63735,OPENALEX
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49340,OPENALEX
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23702,OPENALEX


In [4]:
print("=== TRANSFORM: PubMed ===")
df_pm = standardize(records_pm, source="pubmed")
print(f"Shape: {df_pm.shape}")
print(f"Columns: {df_pm.columns.tolist()}")
df_pm[['AU', 'TI', 'PY', 'SO', 'TC', 'DB']].head(3)

=== TRANSFORM: PubMed ===
Shape: (10, 26)
Columns: ['UT', 'TI', 'SO', 'JI', 'PY', 'VL', 'IS', 'LA', 'DT', 'RP', 'AU', 'AF', 'DI', 'PMID', 'BP', 'EP', 'CR', 'AB', 'C1', 'AU_CO', 'DE', 'ID', 'TC', 'DB', 'SR', 'SR_FULL']


,AU,TI,PY,SO,TC,DB
0,[Scharf C],Astrobiology in the Time of Artificial Intelli...,2026,Astrobiology,0,PUBMED
1,"[Wu H, Wang Y, Song J, Yan B, Wang C, Liu J, H...",Expression of the High Affinity Neurotensin Re...,2026,Annals of surgery,0,PUBMED
2,"[Marra JD, Zini G]",Next Generation Digital Morphology: Blast Prec...,2026,International journal of laboratory hematology,0,PUBMED


### Inspect multi-value fields
Author keywords (`DE`) and cited references (`CR`) must be `list[str]`.

In [5]:
print("=== Multi-value fields (OpenAlex) ===")
print(f"AU type: {type(df_oa['AU'].iloc[0])}")
print(f"AU sample: {df_oa['AU'].iloc[0]}")
print(f"\nDE type: {type(df_oa['DE'].iloc[0])}")
print(f"DE sample: {df_oa['DE'].iloc[0]}")
print(f"\nCR type: {type(df_oa['CR'].iloc[0])}")
print(f"CR sample (first 2): {df_oa['CR'].iloc[0][:2]}")

=== Multi-value fields (OpenAlex) ===
AU type: <class 'list'>
AU sample: ['Fabián Pedregosa', 'Gaël Varoquaux', 'Alexandre Gramfort', 'Vincent Michel', 'Bertrand Thirion', 'Olivier Grisel', 'Mathieu Blondel', 'Müller, Andreas', 'Nothman, Joel', 'Louppe, Gilles', 'Peter Prettenhofer', 'Ron J. Weiss', 'Vincent Dubourg', 'Jake Vanderplas', 'Alexandre Passos', 'David Cournapeau', 'Matthieu Brucher', 'Matthieu Perrot', 'Édouard Duchesnay']

DE type: <class 'list'>
DE sample: ['Python (programming language)', 'Documentation', 'Computer science', 'MIT License', 'Artificial intelligence', 'Machine learning', 'Programming language', 'License', 'Software engineering', 'Operating system']

CR type: <class 'list'>
CR sample (first 2): ['Chang C, 2011, ACM TRANSACTIONS ON INTELLIGENT SYSTEMS AND TECHNOLOGY', 'Friedman J, 2010, PUBMED']


---
## PHASE 3: VALIDATE
The validation module checks:
1. All mandatory columns exist
2. No NaN or None values remain
3. Multi-value columns are correctly typed as lists

In [6]:
from www.services.validator import validate

print("=== VALIDATE: OpenAlex ===")
df_oa = validate(df_oa)
print(f"\nSR sample: {df_oa['SR'].iloc[0]}")
df_oa[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: OpenAlex ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Pedregosa F, 2012, ARXIV (CORNELL UNIVERSITY)


,SR,DE,AB
0,"Pedregosa F, 2012, ARXIV (CORNELL UNIVERSITY)","[Python (programming language), Documentation,...",Scikit-learn is a Python module integrating a ...
1,"NA, 1989, CHOICE REVIEWS ONLINE","[Computer science, Artificial intelligence, Ma...",From the Publisher:\r\nThis book brings togeth...
2,"Quinlan J, 1992,","[Computer science, Unix, Classifier (UML), Mac...",Classifier systems play a major role in machin...


In [7]:
print("=== VALIDATE: PubMed ===")
df_pm = validate(df_pm)
print(f"\nSR sample: {df_pm['SR'].iloc[0]}")
df_pm[['SR', 'DE', 'AB']].head(3)

=== VALIDATE: PubMed ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.

SR sample: Scharf C, 2026, ASTROBIOLOGY


,SR,DE,AB
0,"Scharf C, 2026, ASTROBIOLOGY",[],
1,"Wu H, 2026, ANN SURG",[],
2,"Marra J, 2026, INT J LAB HEMATOL",[],


---
## FULL PIPELINE — 200 records
End-to-end demonstration with 200 records per platform, exported to CSV.

In [8]:
from www.services.io_utils import save_standardized_csv, load_standardized_csv, LIST_COLUMNS, STR_COLUMNS


In [9]:
print("=== FULL PIPELINE: OpenAlex (200 records) ===")
records_oa_200 = retrieve(query="machine learning", platform="openalex", total=200)
df_oa_200 = standardize(records_oa_200, source="openalex")
df_oa_200 = validate(df_oa_200)
save_standardized_csv(df_oa_200, "test_openalex_200.csv")  # ";" delimiter for multi-value fields, as required by the spec
print(f"Shape: {df_oa_200.shape}")
print("CSV saved: test_openalex_200.csv")
df_oa_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)


=== FULL PIPELINE: OpenAlex (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 26)
CSV saved: test_openalex_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Fabián Pedregosa, Gaël Varoquaux, Alexandre G...",Scikit-learn: Machine Learning in Python,2012,ARXIV (CORNELL UNIVERSITY),63735,"Pedregosa F, 2012, ARXIV (CORNELL UNIVERSITY)"
1,[],"Genetic algorithms in search, optimization, an...",1989,CHOICE REVIEWS ONLINE,49340,"NA, 1989, CHOICE REVIEWS ONLINE"
2,[J. R. Quinlan],C4.5: Programs for Machine Learning,1992,,23702,"Quinlan J, 1992,"
3,[Arthur Asuncion],UCI Machine Learning Repository,2007,MEDICAL ENTOMOLOGY AND ZOOLOGY,24350,"Asuncion A, 2007, MEDICAL ENTOMOLOGY AND ZOOLOGY"
4,"[Ian H. Witten, Eibe Frank]",Data Mining: Practical Machine Learning Tools ...,2011,ELSEVIER EBOOKS,25716,"Witten I, 2011, ELSEVIER EBOOKS"
5,[Nasser M. Nasrabadi],Pattern Recognition and Machine Learning,2007,JOURNAL OF ELECTRONIC IMAGING,22083,"Nasrabadi N, 2007, JOURNAL OF ELECTRONIC IMAGING"
6,[David E. Goldberg],"Genetic Algorithms in Search, Optimization and...",1988,,17773,"Goldberg D, 1988,"
7,[],Proceedings of the 24th international conferen...,2007,,11733,"NA, 2007,"
8,"[Carl Edward Rasmussen, Christopher K. I. Will...",Gaussian Processes for Machine Learning,2005,THE MIT PRESS EBOOKS,10489,"Rasmussen C, 2005, THE MIT PRESS EBOOKS"
9,[Kevin P. Murphy],Machine learning a probabilistic perspective,2012,,9328,"Murphy K, 2012,"


In [10]:
print("=== FULL PIPELINE: PubMed (200 records) ===")
records_pm_200 = retrieve(query="machine learning", platform="pubmed", total=200, mindate="2015", maxdate="2024")
df_pm_200 = standardize(records_pm_200, source="pubmed")
df_pm_200 = validate(df_pm_200)
save_standardized_csv(df_pm_200, "test_pubmed_200.csv")  # ";" delimiter for multi-value fields, as required by the spec
print(f"Shape: {df_pm_200.shape}")
print("CSV saved: test_pubmed_200.csv")
df_pm_200[['AU', 'TI', 'PY', 'SO', 'TC', 'SR']].head(10)


=== FULL PIPELINE: PubMed (200 records) ===
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.
Shape: (200, 26)
CSV saved: test_pubmed_200.csv


,AU,TI,PY,SO,TC,SR
0,"[Figueroa-Quiñones J, Ipanaque-Neyra J, Gómez ...","Development, validation and use of artificial-...",2023,F1000Research,0,"Figueroa-Quiñones J, 2023, F1000RES"
1,"[de Mattos BP, Mattjie C, Ravazio R, Barros RC...",Craving for a Robust Methodology: A Systematic...,2026,International journal of mental health and add...,0,"de Mattos B, 2026, INT J MENT HEALTH ADDICT"
2,"[Kuang A, Yu Y, Siddique J, Scholtens D]",Imputation of Missing Continuous Glucose Monit...,2026,Journal of diabetes science and technology,0,"Kuang A, 2026, J DIABETES SCI TECHNOL"
3,"[Marsico C, Renteria C, Grimm JR, Fernandez-Ar...",A Machine Learning Approach to Quantitative An...,2025,Small structures,0,"Marsico C, 2025, SMALL STRUCT"
4,"[Upadhyay K, Fuhg JN, Bouklas N, Ramesh KT]",Physics-informed data-driven discovery of cons...,2026,Computational mechanics,0,"Upadhyay K, 2026, COMPUT MECH"
5,"[Zhan Z, Zhou S, Deng J, Zhang R]",Improving electronic health record processing ...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Zhan Z, 2024, AMIA ANNU SYMP PROC"
6,"[Xie Y, Cui H, Zhang Z, Lu J, Shu K, Nahab F, ...",KERAP: A Knowledge-Enhanced Reasoning Approach...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Xie Y, 2024, AMIA ANNU SYMP PROC"
7,"[Sivarajkumar S, Ameri K, Li C, Wang Y, Jiang M]",Automating Adjudication of Cardiovascular Even...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Sivarajkumar S, 2024, AMIA ANNU SYMP PROC"
8,"[Wang M, Kuan YH, Alba PR, Gan Q, Schoen MW, T...",Developing Large Language Model-based Pipeline...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Wang M, 2024, AMIA ANNU SYMP PROC"
9,"[Nguyen QN, Wu H, Pontikos N, Wang SY]",Addressing Generalizability in Clinical Named ...,2024,AMIA ... Annual Symposium proceedings. AMIA Sy...,0,"Nguyen Q, 2024, AMIA ANNU SYMP PROC"


---
## PHASE 4: CSV ROUND-TRIP INTEGRITY
The CSVs just saved are now reloaded from disk to demonstrate that the type contract
survives a round trip through a file (e.g. when the CSV is manually re-imported into the
dashboard via the "Import Raw Data" tab).

Without this safeguard, `pandas.read_csv()` would treat empty cells as `NaN` instead
of as an empty string `""`, and the list columns (`AU`, `CR`, `DE`, ...) would come back
as plain strings instead of `list[str]` — silently breaking the type contract
enforced by `standardize()` and `validate()`.

**Note:** this does not affect the live flow (API Query → dashboard), which always stays
in memory and never goes through a CSV file — it only concerns the standalone CSV
required as a separate deliverable by the spec ("Provide in output a standardized CSV file").


In [11]:
print("=== ROUND-TRIP: OpenAlex CSV reloaded from disk ===")
df_oa_reloaded = load_standardized_csv("test_openalex_200.csv")
print(f"AU   type: {type(df_oa_reloaded['AU'].iloc[0])} | sample: {df_oa_reloaded['AU'].iloc[0][:2]}")
print(f"PMID type: {type(df_oa_reloaded['PMID'].iloc[0])} | value: {repr(df_oa_reloaded['PMID'].iloc[0])}")
print(f"TC   type: {type(df_oa_reloaded['TC'].iloc[0])}")
print(f"PY   type: {type(df_oa_reloaded['PY'].iloc[0])} | value: {df_oa_reloaded['PY'].iloc[0]}")
df_oa_reloaded = validate(df_oa_reloaded)

=== ROUND-TRIP: OpenAlex CSV reloaded from disk ===
AU   type: <class 'list'> | sample: ['Fabián Pedregosa', 'Gaël Varoquaux']
PMID type: <class 'str'> | value: ''
TC   type: <class 'numpy.int64'>
PY   type: <class 'str'> | value: 2012
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.


In [12]:
print("=== ROUND-TRIP: PubMed CSV reloaded from disk ===")
df_pm_reloaded = load_standardized_csv("test_pubmed_200.csv")

print(f"AU   type: {type(df_pm_reloaded['AU'].iloc[0])}")
print(f"UT   type: {type(df_pm_reloaded['UT'].iloc[0])}")
print(f"PMID type: {type(df_pm_reloaded['PMID'].iloc[0])}")

df_pm_reloaded = validate(df_pm_reloaded)

=== ROUND-TRIP: PubMed CSV reloaded from disk ===
AU   type: <class 'list'>
UT   type: <class 'str'>
PMID type: <class 'str'>
Running validation...
  PASS — all mandatory columns present
  PASS — no null values found
  PASS — all column types correct
Validation passed.


In [13]:
df_oa_200.to_excel("test_openalex_200.xlsx", index=False) 

df_pm_200.to_excel("test_pubmed_200.xlsx", index=False)  


---
## Summary

| Platform | Records | Columns | NaN | SR | CSV round-trip |
|----------|---------|---------|-----|----|----------------|
| OpenAlex | 200 | 26 | 0 | ✅ | ✅ |
| PubMed   | 200 | 26 | 0 | ✅ | ✅ |

The ETL pipeline successfully:
- Extracted data from OpenAlex and PubMed REST APIs
- Transformed raw JSON into the WoS standard schema
- Enforced type contracts (list[str], str, int)
- Validated all mandatory columns
- Generated standardized CSV files ready for Bibliometrix-Python analysis
- Verified that the type contract survives a full save-to-disk / reload-from-disk cycle via `load_standardized_csv()`